# 🛞 Notebook 2: Sidecar as a separate process

In real deployments the sidecar is its **own process** on the same host/pod.
Here we simulate that with two Python scripts talking over a local HTTP port.

## 🛠️ Setup

```bash
cd 05-microservices/sidecar
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## App: dumb, trusts only localhost

In [ ]:
# --- Imagine this in app.py ---
from http.server import BaseHTTPRequestHandler, HTTPServer
import threading, time, json, urllib.request

class App(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200)
        self.send_header('Content-Type', 'application/json')
        self.end_headers()
        self.wfile.write(json.dumps({'msg':'hello from app','path':self.path}).encode())
    def log_message(self, *a, **kw): pass

def run_app():
    HTTPServer(('127.0.0.1', 9001), App).serve_forever()

threading.Thread(target=run_app, daemon=True).start()
time.sleep(0.3)
print('app up on 9001')


## Sidecar: listens on 9000, adds auth + logs, forwards to 9001

In [ ]:
class Sidecar(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.headers.get('X-Token') != 'secret':
            self.send_response(401); self.end_headers(); self.wfile.write(b'nope'); return
        t0 = time.time()
        with urllib.request.urlopen(f'http://127.0.0.1:9001{self.path}') as r:
            body = r.read()
        print(f'[sidecar] {self.path} {time.time()-t0:.3f}s')
        self.send_response(200); self.end_headers(); self.wfile.write(body)
    def log_message(self, *a, **kw): pass

def run_sc():
    HTTPServer(('127.0.0.1', 9000), Sidecar).serve_forever()

threading.Thread(target=run_sc, daemon=True).start()
time.sleep(0.3)
print('sidecar up on 9000')


In [ ]:
import urllib.request, urllib.error

def call(token):
    req = urllib.request.Request('http://127.0.0.1:9000/ping', headers={'X-Token': token})
    try:
        return urllib.request.urlopen(req).read().decode()
    except urllib.error.HTTPError as e:
        return f'HTTP {e.code}'

print('good token:', call('secret'))
print('bad token :', call('wrong'))


### Observations
- The **app never checked auth** — the sidecar did.
- The app listens only on `127.0.0.1` so no one can bypass the sidecar.
- Swap the sidecar independently (new TLS cert? new log format?) without redeploying the app.

This is what Envoy + Istio do at scale.